# AAROH Multimodal Fusion

Train the fusion head over tabular, text, audio, longitudinal, and behavioral representations. The output is a normalized representation and modality weights only.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO_DIR = '/content/AAROH'
if not os.path.isdir(REPO_DIR): raise FileNotFoundError(f'Expected repository at {REPO_DIR}')
os.chdir(REPO_DIR)

In [ ]:
%pip install -q torch transformers scikit-learn
import torch
if not torch.cuda.is_available(): raise RuntimeError('A CUDA runtime is required for fusion training')
print('CUDA:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/AAROH')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints' / 'fusion'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
Path('models/multimodal_fusion').mkdir(parents=True, exist_ok=True)

In [ ]:
!python -m backend.ml.training.train_multimodal_fusion \
  --data-dir datasets/processed \
  --output-dir models/multimodal_fusion \
  --checkpoint-dir checkpoints/fusion \
  --drive-checkpoint-dir $CHECKPOINT_DIR \
  --epochs 5 \
  --batch-size 16 \
  --fp16

In [ ]:
from backend.ml.training.models.fusion.model import MultimodalFusionModel
from backend.ml.training.models.fusion.dataset import build_synthetic_multimodal_records
model = MultimodalFusionModel.load_from_artifact('models/multimodal_fusion', device='cuda')
result = model.fuse(build_synthetic_multimodal_records(count=1, seed=42)[0])
assert len(result['fused_embedding']) == 256
assert abs(sum(result['modality_weights'].values()) - 1.0) < 1e-3
print('Fusion artifact reload and inference verification passed')

In [ ]:
from pathlib import Path
required = ['config.json','metadata.json','weights','metrics.json','modality_schema.json']
missing = [name for name in required if not (Path('models/multimodal_fusion') / name).exists()]
if missing: raise FileNotFoundError(missing)
print('Multimodal Fusion artifacts exported:', required)